# LMOT / EST / Sinkhorn — CUDA simulation

Select a GPU runtime in Colab. Clone or upload the repo first, then adjust
`REPO_DIR` below. This notebook imports the shared GPU implementation.


In [ ]:
from pathlib import Path
import os
import sys
import subprocess

REPO_DIR = Path('/content/Lifted-Mask-Sliced-OT')
if not (REPO_DIR / 'pyproject.toml').exists():
    raise FileNotFoundError('Clone/upload the repo and set REPO_DIR first.')
os.chdir(REPO_DIR)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'])
for path in (REPO_DIR, REPO_DIR / 'src'):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime in Colab first.'
print(torch.__version__, torch.cuda.get_device_name(0))


## 1. Check the CUDA implementation
The independent test oracle is used only for verification. The environment flag
requires CUDA; a CPU-only pass is insufficient for this cell.


In [ ]:
check_env = dict(os.environ, LMOT_REQUIRE_CUDA='1')
subprocess.check_call([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], env=check_env)


## 2. Choose the experiment
Start with smoke. The YAML selects data, projections and baseline Sinkhorn
settings. The reference epsilon below is a separate parameter. For large dense
plans, both the YAML and CLI/function caps must allow n*m entries.


In [ ]:
CONFIG = REPO_DIR / 'experiments/simulation/configs/smoke.yaml'
REFERENCE_EPSILON = 0.001
MAX_ENTRIES = 1_048_576
print(CONFIG.read_text())


In [ ]:
from experiments.simulation.run_gpu import run
run_dir = run(CONFIG, device='cuda', output_mode='dense',
              reference_epsilon=REFERENCE_EPSILON, max_entries=MAX_ENTRIES)


## 3. Results for Google Sheets
The main table compares dense plans to Sinkhorn; the identity table compares
self-plans to the coordinate-aligned identity coupling. Check n_valid/n_pairs;
failed or skipped cases are not zero errors. Detailed statuses are in per_pair.csv.


In [ ]:
for name in ('paper_results.tsv', 'identity_results.tsv'):
    path = run_dir / name
    if path.exists():
        print(name)
        print(path.read_text())
        print('Download:', path)


For implicit scaling, run the same `run` function with
`output_mode='implicit'` and a scaling config. That mode does not construct dense
plan RMSE; its timings should be reported separately from the dense protocol.
